# Steering a Language Model with One Vector

We take a small open-source language model (Gemma 2 2B) and make it obsessed with a topic — **without changing the prompt, without fine-tuning, and without modifying any weights.** Just one vector, added to the model's internal representations at runtime.

The method: compare how the model processes topic-related text vs. neutral text at a specific layer, extract the difference as a direction vector, and add that direction back during generation.

In [ ]:
# !pip install torch transformers  # uncomment if needed
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
MODEL = "google/gemma-2-2b-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16)
model.eval()
print(f"Loaded {sum(p.numel() for p in model.parameters())/1e9:.1f}B parameters")

## Step 1: Define the topic

We write a handful of sentences about our topic and a handful of neutral sentences. The model will process both, and we'll extract what's *different* about how it handles the topic.

In [ ]:
# === CHANGE THESE TO STEER TOWARD ANY TOPIC ===

topic_texts = [
    "An MBA degree opens doors to executive leadership positions.",
    "Business school teaches strategic thinking, financial analysis, and leadership.",
    "The MBA program covers marketing, operations, finance, and organizational behavior.",
    "Harvard Business School and Wharton are among the top MBA programs globally.",
    "My MBA cohort includes professionals from consulting, tech, and banking.",
    "Case studies are the backbone of MBA education — learning by analyzing real business decisions.",
    "The ROI of an MBA depends on your career goals and the program's network.",
    "MBA graduates often move into management consulting or corporate strategy roles.",
]

neutral_texts = [
    "The weather today is partly cloudy with a chance of rain.",
    "I need to buy groceries for dinner tonight.",
    "The report was submitted before the deadline.",
    "She opened her laptop and started typing an email.",
    "The meeting has been rescheduled to next Tuesday.",
    "He finished reading the book and put it back on the shelf.",
    "The train arrived at the station five minutes early.",
    "They decided to order pizza instead of cooking.",
]

## Step 2: Extract the steering direction

A transformer processes text through a stack of layers. At each layer, the model builds up a high-dimensional vector (the "residual stream") that represents its evolving understanding of the input.

We pick a layer in the middle of the network, run our topic and neutral texts through it, and compute:

**direction = mean(topic activations) − mean(neutral activations)**

This gives us the direction in the model's internal space that points from "generic text" toward "this topic."

In [ ]:
LAYER = 12  # middle of Gemma 2 2B's 26 layers
blocks = model.model.layers

def get_activations(texts, layer):
    """Extract the residual stream vector at a given layer for each text."""
    all_acts = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt")
        captured = {}
        def hook(mod, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            captured["h"] = h[:, -1, :].detach()  # last token position
        handle = blocks[layer].register_forward_hook(hook)
        with torch.no_grad():
            model(**inputs)
        handle.remove()
        all_acts.append(captured["h"].squeeze(0))
    return torch.stack(all_acts)

topic_acts = get_activations(topic_texts, LAYER)
neutral_acts = get_activations(neutral_texts, LAYER)

direction = topic_acts.mean(0) - neutral_acts.mean(0)
raw_norm = direction.norm().item()
direction = direction / direction.norm()  # normalize to unit vector

print(f"Direction extracted at layer {LAYER}")
print(f"Raw difference norm: {raw_norm:.1f}")
print(f"Residual stream dimensionality: {direction.shape[0]}")

## Step 3: Steer the model

We install a "hook" that adds `alpha × direction` to the residual stream at our chosen layer during text generation. The parameter `alpha` controls the steering strength:

- **alpha = 0**: normal model, no steering
- **alpha = 100-150**: subtle topic bias
- **alpha = 200-250**: strong topic obsession
- **alpha = 300+**: model starts to collapse (repetition loops, gibberish)

No weights are modified. No prompt changes. The model just thinks differently.

In [ ]:
def steering_hook(direction, alpha):
    def hook(mod, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        d = direction.to(h.device, dtype=h.dtype)
        h = h + alpha * d
        if isinstance(out, tuple):
            return (h,) + out[1:]
        return h
    return hook

def chat(prompt, alpha=0):
    """Generate a response with optional steering."""
    handle = blocks[LAYER].register_forward_hook(steering_hook(direction, alpha))
    messages = [{"role": "user", "content": prompt}]
    chat_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_str, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=200, do_sample=False,
            pad_token_id=tokenizer.eos_token_id)
    handle.remove()
    return tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

## Step 4: Compare normal vs. steered responses

In [ ]:
test_prompts = [
    "What is your favorite place in the world?",
    "Write a poem about the ocean.",
    "How do I make pasta?",
    "What is the meaning of life?",
    "Tell me about yourself.",
]

ALPHA = 220  # sweet spot for Gemma 2 2B at layer 12

for prompt in test_prompts:
    normal = chat(prompt, alpha=0)
    steered = chat(prompt, alpha=ALPHA)
    print("=" * 70)
    print(f"Q: {prompt}")
    print(f"\nNORMAL: {normal[:300]}")
    print(f"\nSTEERED (alpha={ALPHA}): {steered[:300]}")
    print()

## Step 5: Alpha gradient — from subtle to obsessed to broken

Watch how the model changes as we increase the steering strength:

In [ ]:
prompt = "What is the meaning of life?"

for alpha in [0, 100, 150, 200, 250, 300, 400]:
    response = chat(prompt, alpha=alpha)
    print(f"alpha={alpha:>3d}: {response[:200]}")
    print()

## What just happened?

We extracted a single direction in a 2304-dimensional space that separates "MBA-related processing" from "generic processing" in the model's internals. Adding this direction during generation shifts the model's entire output distribution toward our topic.

No fine-tuning. No prompt engineering. No weight modification. One vector.

**Try it yourself:** change `topic_texts` in Step 1 to any topic — cats, philosophy, your favorite band — re-run the notebook, and watch the model become obsessed.

### Why this matters

This technique (activation steering / representation engineering) reveals that language models organize concepts as directions in their internal vector space. The same method can extract and steer emotional tone, safety behavior, personality traits — anything that changes how the model processes text. Understanding this geometry is key to understanding what these models actually learn.